In [ ]:
import json
import os
import torch
from tqdm import tqdm
import random
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# # !!!! ALERT: TO REMOVE WHEN INTEGRATING 02 INTO MAIN !!!
# !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git
# %cd Project_Generative_AI_for_Data_Augmentation

Cloning into 'Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 97 (delta 41), reused 44 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 6.84 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [ ]:
import os
PROJECT_ROOT = os.getcwd()

In [ ]:
PROJECT_ROOT

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# PROJECT_ROOT = (
#     "/content/drive/MyDrive/Colab Notebooks/"
#     "ProfessionAI_AIengineering/9. Generative AI/"
#     "Project_Generative_AI"
# )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")
os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(CAPTIONS_DIR, "captions_train_small.json")
TEXT_VARIATION_FILE = os.path.join(TEXT_VARIATIONS_DIR, "text_variations_train_small.json")

In [ ]:
# # !!!! ALERT: TO REMOVE WHEN INTEGRATING 02 INTO MAIN !!!
# INSTALL_DEPS = True   # set to False after first successful run

# if INSTALL_DEPS:
#     !pip install -r "$PROJECT_ROOT/requirements.txt"

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Captions

In [ ]:
with open(CAPTION_FILE, "r") as f:
  captions_dict = json.load(f)

In [ ]:
# captions_dict['981']['captions']

In [ ]:
# captions_dict['1022']

# Model Exploration

## FLAN-T5-Large Model

In [ ]:
model_name = "google/flan-t5-large"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

model.eval()

In [ ]:
def build_prompt(caption):
  return f"Rewrite this caption in three different ways: {caption}"

In [ ]:
def generate_variations(prompt, num_variations=3):
  inputs = tokenizer(prompt, return_tensors = "pt", truncation=True).to(device)

  outputs = model.generate(**inputs,
                          max_new_tokens=40,
                          # do_sample=True,
                          # temperature=0.7,
                          # top_k=50,
                          # top_p=0.9,
                          # # num_beams=5,
                          # num_return_sequences=num_variations

                          # num_beams=6,
                          # num_beam_groups=3,
                          # diversity_penalty=0.5,
                          # num_return_sequences=num_variations,
                          # early_stopping=True,
                          # trust_remote_code=True
                          do_sample=True,
                          temperature=0.6,
                          top_p=0.85,
                          num_return_sequences=num_variations,
                          pad_token_id=tokenizer.eos_token_id
                          )

  decoded = [
      tokenizer.decode(output, skip_special_tokens=True)
      for output in outputs
  ]

  return decoded

In [ ]:
for img, data in list (captions_dict.items())[:10]:
  captions = list(set(data["captions"])) # remove duplicates

  for caption in captions:
    prompt = build_prompt(caption)
    # print(prompt)
    generated = generate_variations(prompt)

    print("Original:", caption)
    print("Generated:", generated)

FLAN-T5-Large was evaluated as a candidate model for caption rewriting.

However, qualitative analysis showed several consistent issues:

- **poor semantic preservation:** generated output often drifted away from original meaning

- **hallucinations:** frequent introduction of unrelated object, people, or scenes

Due to these limitations, FLAN-T5-Large was deemed unsuitable for controlled data augmentation.

## FLAN-T5-XL Model

In [ ]:
model_name = "google/flan-t5-xl"
tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,     # REQUIRED for T4
    device_map="auto",             # automatic GPU placement
    low_cpu_mem_usage=True
)

model.eval()

In [ ]:
def build_prompt(caption):
    return f"Rewrite this caption in three different ways: {caption}"

In [ ]:
def generate_variations(prompt, num_variations=3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=5,
        num_return_sequences=num_variations,
        early_stopping=True
        # do_sample=True,
        # temperature=0.7,
        # top_k=50,
        # top_p=0.9,
        # # num_beams=5,
        # num_return_sequences=num_variations
    )

    decoded = [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

    return decoded

In [ ]:
for img, data in list (captions_dict.items())[:10]:
  captions = list(set(data["captions"])) # remove duplicates

  for caption in captions:
    prompt = build_prompt(caption)
    # print(prompt)
    generated = generate_variations(prompt)

    print("Original:", caption)
    print("Generated:", generated)

FLAN-T5-XL produced grammatically correct and semantically faithful rewrites.

However, the generated variations showed very low lexical diversity, often resulting in near-duplicate sentences with only minor wording or punctuation changes.

Since the goal of this stage is meaningful data augmentation, higher variation diversity was required.

## Mistral 7B Instruct Model

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Define 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()


In [ ]:
!nvidia-smi

In [ ]:
def build_prompt(caption):
    return f"""<s>[INST]
Rewrite the caption in two different ways.
Keep the meaning the same.

Caption: {caption}
[/INST]"""

In [ ]:
def generate_variations(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
      outputs = model.generate(
          **inputs,
          max_new_tokens=40,
          do_sample=True,
          temperature=0.7,
          top_p=0.9,
          num_return_sequences=1,
          pad_token_id=tokenizer.eos_token_id
      )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = text.split("[/INST]")[-1].strip()

    # Split numbered lines into clean list
    variations = []
    for line in response.split("\n"):
        line = line.strip()
        if len(line) > 5:
            if line[0].isdigit():
                line = line.split(".", 1)[-1].strip()
            variations.append(line)

    return variations[:2]  # ensure exactly 2


In [ ]:
# test first 10
text_variations = {}

items = list(captions_dict.items())

for img, data in tqdm(items[:10]):

    class_name = data["class_name"]
    captions = list(set(data["captions"]))  # remove duplicates

    all_generated = []

    for caption in captions:
        prompt = build_prompt(caption)
        variations = generate_variations(prompt)
        all_generated.extend(variations)

    # Remove duplicates across captions
    all_generated = list(set(all_generated))

    text_variations[img] = {
        "class_name": class_name,
        "original_captions": captions,
        "generated_captions": all_generated
    }


In [ ]:
# # full Loop
# text_variations = {}

# for img, data in tqdm(captions_dict.items()):

#     class_name = data["class_name"]
#     captions = list(set(data["captions"]))  # remove duplicates

#     all_generated = []

#     for caption in captions:
#         prompt = build_prompt(caption)
#         variations = generate_variations(prompt)
#         all_generated.extend(variations)

#     # Remove duplicates across captions
#     all_generated = list(set(all_generated))

#     text_variations[img] = {
#         "class_name": class_name,
#         "original_captions": captions,
#         "generated_captions": all_generated
#     }


In [ ]:
text_variations

In [ ]:
import json

with open(TEXT_VARIATION_FILE, "w") as f:
    json.dump(text_variations, f, indent=4)

Consider
- executing in batch
- add save checkpoints

- Add comment on Mistral
- Run Mistral for all items

In [ ]:
# ! git config --global user.email "fellinegiorgia@gmail.com"
# ! git config --global user.name "Jorj91"

# !git remote remove origin

# !git remote add origin https://MY_TOKEN@github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git

# ! git push --set-upstream origin main

# ! git add .

# ! git commit -m "adapt 02 notebook code for integration in main"

# ! git push

Branch 'main' set up to track remote branch 'main' from 'origin'.
Everything up-to-date
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
